In [7]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(
    model="embeddinggemma:300m",    
    base_url="http://host.docker.internal:11434"
)
embedding.embed_query("대한민국")


[-0.19437765,
 -0.011959986,
 0.052749917,
 -0.0032229896,
 0.06763272,
 0.05428981,
 0.009312452,
 -0.0107384855,
 0.054458104,
 -0.04355588,
 0.004882732,
 0.0023202982,
 -0.01305171,
 0.0035114654,
 0.08511771,
 -0.07794391,
 0.039034974,
 -0.009526386,
 -0.045393705,
 0.042828154,
 0.044899408,
 -0.01685659,
 0.025119929,
 0.00867531,
 -0.00821093,
 -0.034432586,
 0.041581858,
 0.032054875,
 -0.03967496,
 0.027529279,
 -0.048199978,
 -0.0042330343,
 0.04082932,
 -0.0073243864,
 -0.03810314,
 0.052638285,
 -0.0111800125,
 5.8966493e-06,
 0.003173931,
 0.029306076,
 -0.034090884,
 0.055999465,
 -0.0103385905,
 -0.025691364,
 0.047904972,
 0.03759947,
 0.013615689,
 -0.05917774,
 -0.0693935,
 -0.010856275,
 -0.03805721,
 0.05149193,
 -0.011317583,
 0.005212514,
 -0.01899921,
 -0.01184779,
 -0.017245524,
 -0.01697456,
 -0.015337082,
 -0.00011496735,
 -0.023440585,
 0.007905432,
 -0.014949185,
 -0.007077219,
 0.027728548,
 -0.044803485,
 -0.0060004736,
 -0.001155925,
 -0.012296313,
 0.1

In [8]:
from langchain_postgres import PGEngine, PGVectorStore
import os
DB_USER = os.getenv("DB_USER", "langchain")
DB_PASSWORD = os.getenv("DB_PASSWORD", "langchain")
DB_HOST = os.getenv("DB_HOST", "postgres")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "langchain")

CONNECTION_STRING = (
    f"postgresql+psycopg://"
    f"{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = PGEngine.from_connection_string(
    url=CONNECTION_STRING,
)

vector_store= PGVectorStore.create_sync(
    engine=engine,
    table_name = "stocks",
    embedding_service=embedding
)


retriever = vector_store.as_retriever(search_type='mmr', 
                                      search_kwargs={
                                        'k' : 10,
                                        'fetch_k' : 30, 
                                      'lambda_mult' : 0.5}
                                      )


In [3]:
rt = retriever.invoke("반도체")


In [4]:
rt

[Document(id='59bc8694-11da-4525-b6b0-98402c999955', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-07-13T08:32:25+09:00', 'title': 'POSCO홀딩스', 'author': '작성자', 'subject': '소제목입니다', 'keywords': '(005490)', 'moddate': '2026-07-13T08:32:25+09:00', 'source': 'pdf/20260713_company_753862000.pdf', 'total_pages': 8, 'page': 4, 'page_label': '5'}, page_content=''),
 Document(id='d6295024-d43c-40bc-94e5-e57832e6c764', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-07-22T14:01:10+09:00', 'title': 'LG유플러스', 'author': '작성자', 'subject': '소제목입니다', 'keywords': '(032640)', 'moddate': '2026-07-22T14:01:10+09:00', 'source': 'pdf/20260723_company_440881000.pdf', 'total_pages': 6, 'page': 2, 'page_label': '3'}, page_content=''),
 Document(id='aa496d7b-c95b-48d0-88d5-73869dae3f12', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-07-13T08:32:

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser


In [4]:
llm = ChatOllama(
   base_url="http://host.docker.internal:11434", # 원격 서버 주소
   model="gemma4:31b-mlx",
   temperature=0.2,
   reasoning=True
)


In [5]:
prompt = ChatPromptTemplate.from_template(
    """
    당신은 유능한 애널리스트입니다. 제시된 질문과 자료를 바탕으로 기업에 대해서 평가하세요.
    관련성이 높은 자료에 집중하고 검토하는 과정을 포함하세요.
    규칙 : 
    1. 제공된 정보에서 이야기 할 것 
    2. 제공된 정보의 출처를 알려줄 것
    {context} 
    
    질문 : {input}

    """
)


In [9]:
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)


In [ ]:
rt = rag_chain.invoke({'input' : "반도체 전망에 대해서 알려줘"})

In [14]:
print(rt['answer'])

분석가로서 답변드립니다.

제시해주신 규칙에 따라 제공된 자료를 바탕으로 분석을 진행해야 하나, **현재 질문과 함께 제공된 참고 자료(데이터)가 없습니다.** 

또한, 요청하신 **'양자 역학'**은 특정 기업에 대한 정보가 아닌 과학 이론에 관한 질문으로, 제가 수행해야 할 '기업 평가'라는 목적과 부합하지 않으며 분석할 근거 자료 또한 제시되지 않았습니다.

**[검토 결과]**
1. **자료 검토:** 제공된 자료 없음.
2. **분석 가능 여부:** 불가 (근거 자료 부재 및 질문 내용이 기업 평가와 무관함).

평가하시고자 하는 **특정 기업의 이름과 관련 자료(사업보고서, 뉴스, 재무제표 등)**를 제공해 주시면, 규칙에 따라 철저히 분석하여 보고해 드리겠습니다.


In [17]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [18]:
history_store = {}

In [19]:
def get_session_history(session_id : str) -> BaseChatMessageHistory:
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()

    return history_store[session_id]


In [20]:
config = {
    'configurable' :{
        'session_id' : 'oracle-01'
    }
}


In [21]:
prompt = ChatPromptTemplate.from_messages(
    [('system', """당신은 유능한 애널리스트입니다. 제시된 질문과 자료를 바탕으로 기업에 대해서 평가하세요.
                    규칙 : 
                    1. 제공된 정보에서 이야기 할 것 
                    2. 제공된 정보의 출처를 알려줄 것
                    {context} 
                """),
     MessagesPlaceholder(variable_name="chat_history"),
     ('human', "{question}")
    ]
    
)


In [22]:
rag_chain = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],
        "chat_history": lambda x: x.get("chat_history", []),
    }
    | prompt
    | llm
)


In [23]:
conversation_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key='question',
    history_messages_key='chat_history'
)


/usr/local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [24]:
answer1 = conversation_chain.invoke(
    {
        'question' : '반도체 증시의 전망?'
    },
    config=config)


In [28]:
answer1.content

"제시된 자료를 바탕으로 분석한 반도체 증시 및 업황의 전망은 매우 긍정적이며, 본격적인 **'업사이클(Upcycle) 구간'**에 진입한 것으로 평가됩니다. 상세 내용은 다음과 같습니다.\n\n### 1. 전반적인 시장 전망: 호황기 진입 및 성장 지속\n*   **실적 기반의 호황:** 반도체 소부장(소재·부품·장비) 기업들의 실적이 사상 최대치를 경신할 것으로 전망되며, 이는 단순한 기대감이 아닌 숫자로 증명되는 호황으로 분석됩니다. (출처: 뉴파워프라즈마 자료)\n*   **성장 기간의 확장:** 메모리 업체의 중단기 증설 속도가 빨라지고 있으며, 2030년 캐파(CAPA) 2배 목표에 따라 **2028년까지 성장세가 이어질 것**으로 보입니다. (출처: 원익 IPS 자료)\n*   **수요 정점의 지연:** 신규 Fab 증설에 따라 플라즈마 수요 등의 정점은 **2027년 이후**가 될 것으로 예상되어, 향후 지속적인 성장이 기대됩니다. (출처: 뉴파워프라즈마 자료)\n\n### 2. 주요 성장 동력 (Key Drivers)\n*   **AI 추론 시대의 도래:** AI 추론 시대가 오면서 메모리 수요가 급증하였고, 이에 따라 삼성전자와 SK하이닉스의 설비투자가 활발히 진행되고 있습니다. (출처: 로체시스템즈 자료)\n*   **투자 영역의 확대:** \n    *   **낸드(NAND) 및 파운드리:** 부진했던 낸드 투자 재개 움직임이 포착되었으며, 파운드리 매출 또한 회복세에 있습니다. (출처: 원익 IPS 자료)\n    *   **글로벌 확장:** 미국 삼성 테일러 팹(PH1) 투자가 하반기에 마무리될 예정이며, 중국 CXMT향 공급 회복 및 상하이 팹 로드맵에 따른 내년 매출 확대가 가시적입니다. (출처: 원익 IPS 자료)\n\n### 3. 주요 기업별 전망 및 지표\n*   **원익 IPS:** 1Q26 수주 잔고가 4,000억 원까지 급증하였으며, 국내 고객사의 증설 계획(디램 기준 150K/M 규모)이 뉴노멀로 자리 잡으며 3Q26부

In [29]:
for chunk in conversation_chain.stream(
    {
        'question' : '내가 무슨 질문 했어?'
    },
    config=config):
    print(chunk.content, end='', flush=True)


사용자님께서는 저에게 **"반도체 증시의 전망?"**에 대해 질문하셨습니다.